In [126]:
import numpy as np

from numba import njit

from scipy.integrate import solve_ivp

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score

import plotly.graph_objects as go

import fastplotlib as fpl;

In [107]:
time = 100
dt = 0.005
steps = int(time / dt)
tau_steps = int(1.5 / dt)

test_size = 0.2
transient_period = int(steps * .05)

t = np.linspace(0, time, steps)

u_val = 1
u = (t.astype(int) % 2) * 2 * u_val - u_val

In [101]:
sigma, rho, beta = 10, 28, 8.0 / 3.0

initial_state = [1.0, 1.0, 1.0]

def lorenz_system(t, state, sigma=sigma, rho=rho, beta=beta):
    x, y, z = state
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return [dx, dy, dz]


sol = solve_ivp(lorenz_system, (0, time), initial_state, t_eval=t)

lorenz_dataset = sol.y.T
lorenz_dataset_after_transient = lorenz_dataset[transient_period:]

In [44]:
fig = go.Figure(
    data=go.Scatter3d(
        x=lorenz_dataset[:, 0],
        y=lorenz_dataset[:, 1],
        z=lorenz_dataset[:, 2],
        mode="lines",
        line=dict(color="blue", width=2),
    )
)

fig.show()

In [30]:
def create_stiffness_matrix(node_positions, connections):
    num_nodes = node_positions.shape[0]
    dims = node_positions.shape[1]
    K = np.zeros((num_nodes * dims, num_nodes * dims))

    for conn in connections:
        node_conn = conn[:2].astype(int)
        node_pos = node_positions[node_conn]
        k_val = conn[2]

        diff_vec = np.diff(node_pos, axis=0).flatten()
        unit_dir = diff_vec / np.linalg.norm(diff_vec)

        sub_block = np.outer(unit_dir, unit_dir)
        k_local = k_val * np.block([[sub_block, -sub_block], [-sub_block, sub_block]])

        global_indices = (node_conn * dims + np.arange(dims)[:, None]).flatten("F")

        for local_row, global_row in enumerate(global_indices):
            for local_col, global_col in enumerate(global_indices):
                K[global_row, global_col] += k_local[local_row, local_col]

    return K

In [ ]:
@njit
def run_simulation(steps, dt, matrix_size, M_INV, C, K, U):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    for i in range(1, steps):
        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + U[i - 1])

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [109]:
def ridge_regression(X, Y):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train_unscaled, X_test_unscaled, Y_train, Y_test = train_test_split(
        X_scaled[tau_steps:], Y[:-tau_steps], test_size=test_size, random_state=42
    )
    X_train = scaler.fit_transform(X_train_unscaled)
    X_test = scaler.transform(X_test_unscaled)

    model = RidgeCV()
    model.fit(X_train, Y_train)
    Y_pred_ridge = model.predict(X_test)

    return model, (Y_test, Y_pred_ridge)

# Chain

In [133]:
N = 10
nodes_pos = np.arange(0, N).reshape(-1, 1)
node_ids = np.arange(N)

In [134]:
rng = np.random.default_rng(42)

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

k_vals = rng.uniform(0.5, 8, size=N-1)

U = np.zeros((steps, matrix_size))
U[:, 0] = u
# U[:, 0] = lorenz_dataset[:, 0]
# U[:, 4] = lorenz_dataset[:, 1]
# U[:, -1] = lorenz_dataset[:, 2]

In [135]:
connections_list = np.column_stack((node_ids[:-1], node_ids[1:], k_vals))

K = create_stiffness_matrix(nodes_pos, connections_list)

In [136]:
disp, v = run_simulation(steps, dt, matrix_size, M_INV, DAMP, K, U)
disp = disp[transient_period:]
v = v[transient_period:]

In [ ]:
disp_reshaped = disp.reshape(steps - transient_period, num_nodes, dims)
disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

nodes_pos_3D = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")

fig = fpl.Figure()
dots_graphic = fig[0, 0].add_scatter(
    data=nodes_pos_3D.astype(np.float32), sizes=15, colors="magenta"
)

spring_lines = []
src_indices = connections_list[:, 0].astype(np.int32)
dst_indices = connections_list[:, 1].astype(np.int32)
for src, dst in zip(src_indices, dst_indices):
    line_coords = np.vstack([nodes_pos_3D[src], nodes_pos_3D[dst]])
    line_graphic = fig[0, 0].add_line(
        data=line_coords.astype(np.float32), thickness=2, colors="cyan"
    )
    line_graphic.colors[:, 3] = 0.2
    spring_lines.append(line_graphic)

frame_tracker = 0
frames_moved = 5
max_frames = 2000
def update_springs(canvas):
    global frame_tracker

    frame_tracker = (frame_tracker + frames_moved) % steps
    print(f"Frame: {frame_tracker}/{steps}", end="\r")

    if frame_tracker >= max_frames:
        canvas.clear_animations()

    new_coords = nodes_pos_3D + disp_3d[frame_tracker]

    dots_graphic.data = new_coords.astype(np.float32)

    for idx, (src, dst) in enumerate(zip(src_indices, dst_indices)):
        src_coord = new_coords[src]
        dst_coord = new_coords[dst]
        spring_lines[idx].data = np.vstack([src_coord, dst_coord]).astype(np.float32)

RFBOutputContext()

In [150]:
def bhbh():
    disp_reshaped = disp.reshape(steps - transient_period, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

    nodes_pos_3D = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")

    fig = fpl.Figure()
    dots_graphic = fig[0, 0].add_scatter(
        data=nodes_pos_3D.astype(np.float32), sizes=15, colors="magenta"
    )

    spring_lines = []
    src_indices = connections_list[:, 0].astype(np.int32)
    dst_indices = connections_list[:, 1].astype(np.int32)
    for src, dst in zip(src_indices, dst_indices):
        line_coords = np.vstack([nodes_pos_3D[src], nodes_pos_3D[dst]])
        line_graphic = fig[0, 0].add_line(
            data=line_coords.astype(np.float32), thickness=2, colors="cyan"
        )
        line_graphic.colors[:, 3] = 0.2
        spring_lines.append(line_graphic)

    frame_tracker = 0
    frames_moved = 5
    max_frames = 2000
    def update_springs(canvas):
        global frame_tracker

        frame_tracker = (frame_tracker + frames_moved) % steps
        print(f"Frame: {frame_tracker}/{steps}", end="\r")

        if frame_tracker >= max_frames:
            canvas.clear_animations()

        new_coords = nodes_pos_3D + disp_3d[frame_tracker]

        dots_graphic.data = new_coords.astype(np.float32)

        for idx, (src, dst) in enumerate(zip(src_indices, dst_indices)):
            src_coord = new_coords[src]
            dst_coord = new_coords[dst]
            spring_lines[idx].data = np.vstack([src_coord, dst_coord]).astype(np.float32)

    
    return (update_springs, fig)

update_springs, fig = bhbh()
fig.add_animations(update_springs)
fig.show()

RFBOutputContext()

In [143]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred) = ridge_regression(
    X, u[transient_period - tau_steps : -tau_steps]
)

In [138]:
r_2 = r2_score(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9427 0.0573


In [139]:
y_pred_discrete = np.where(Y_pred > 0.0, u_val, -u_val)

r_2 = r2_score(Y_test, Y_pred)
accuracy = accuracy_score(Y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.9427 99.47%


In [140]:
weights = model.coef_
labels = [
    f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
]

fig = go.Figure(
    data=[
        go.Bar(
            x=labels,
            y=weights,
            marker_color=np.where(
                weights >= 0, "royalblue", "firebrick"
            ),
        )
    ]
)

fig.update_layout(
    title="Reservoir Node Contribution (Feature Weights)",
    xaxis_title="Spring/Mass Node",
    yaxis_title="Weight Value",
    template="plotly_white",
)

fig.show()